In [10]:
# !pip install torch torchvision torchaudio


In Transformer multi-head attention, two key hyperparameters are:
- `d_model` is the dimension of the token embedding used throughout the transformer. Size of the vector representing each token.
- `num_heads` is the number of attention heads. Splits the embedding into multiple smaller attention mechanisms. 
They control the embedding size and how attention is split across multiple heads. Each head attends to different aspects to the sequence

`num_heads=8` means the model runs 8 attention mechanisms in parallel. 

`d_head = d_model / num_heads`

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0 

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads 

        self.W_q = nn.Linear(d_model, d_model) 
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model) 

        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forwards(self, x, mask=None):
        batch_size, seq_len, _ = x.shape 

        # 1. Project input into Q, K, V
        Q = self.W_q(x) #(B, L, D)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. Split into multiple heads 
        # (B, L, D) -> (B, L, H, Hd) -> (B, H, L, Hd)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Scaled dot-product attention
        # scores: (B, H, L, L)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 4. Weighted sum of values 
        context = torch.matmul(attn_weights, V) 

        # 5. Concatenate heads
        # (B, H, L, Hd) -> (B, L, H, Hd) -> (B, L, D)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # 5. Final output projection
        out = self.W_o(context) #(B, L, D)
        return out
        

Linear(in_features=3, out_features=3, bias=True)